In [1]:
import pandas as pd
import time

print("Downloading and compiling La Liga detailed data...")

# The URLs for La Liga (SP1) from 2010 to 2024
# Format is season start/end year (e.g., 2324 = 2023/2024 season)
seasons = [
    "0506", "0607", "0708", "0809", "0910", 
    "1011", "1112", "1213", "1314", "1415", 
    "1516", "1617", "1718", "1819", "1920", 
    "2021", "2122", "2223", "2324", "2425", "2526"
]

all_seasons_data = []

for season in seasons:
    url = f"https://www.football-data.co.uk/mmz4281/{season}/SP1.csv"
    try:
        print(f"Fetching Season {season}...")
        df = pd.read_csv(url, on_bad_lines='skip')
        df['Season'] = season
        all_seasons_data.append(df)
        time.sleep(1) 
    except Exception as e:
        print(f"Could not load {season}: {e}")

# Combine all the seasons into one massive DataFrame
compiled_laliga = pd.concat(all_seasons_data, ignore_index=True)

# We only care about columns that your feature pipeline actually uses.
# We will filter to keep the core match stats and the B365 odds.
core_columns = [
    'Season', 'Date', 'HomeTeam', 'AwayTeam', 
    'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR',
    'HS', 'AS', 'HST', 'AST', 'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR',
    'B365H', 'B365D', 'B365A'
]

# Keep only the columns that exist (some older seasons might miss a minor column)
available_columns = [c for c in core_columns if c in compiled_laliga.columns]
compiled_laliga = compiled_laliga[available_columns]

# Drop any rows where the Match Result (FTR) is completely missing (empty rows at end of CSVs)
compiled_laliga = compiled_laliga.dropna(subset=['FTR'])

# Rename the columns so they perfectly match what your LiveMatchFeatureEngineer expects
compiled_laliga = compiled_laliga.rename(columns={
    'HS': 'HomeShots', 'AS': 'AwayShots',
    'HST': 'HomeShotsOnTarget', 'AST': 'AwayShotsOnTarget',
    'HC': 'HomeCorners', 'AC': 'AwayCorners',
    'HF': 'HomeFouls', 'AF': 'AwayFouls',
    'HY': 'HomeYellowCards', 'AY': 'AwayYellowCards',
    'HR': 'HomeRedCards', 'AR': 'AwayRedCards',
    'FTHG': 'FullTimeHomeGoals', 'FTAG': 'FullTimeAwayGoals', 'FTR': 'FullTimeResult'
})

# Save it to your data folder
save_path = "data/LaLiga_Detailed.csv"
compiled_laliga.to_csv(save_path, index=False)

print(f"✅ Success! Compiled {len(compiled_laliga)} matches with full stats and odds.")
print(f"Saved to: {save_path}")

Fetching Season 0506...
Fetching Season 0607...
Fetching Season 0708...
Fetching Season 0809...
Fetching Season 0910...
Fetching Season 1011...
Fetching Season 1112...
Fetching Season 1213...
Fetching Season 1314...
Fetching Season 1415...
Fetching Season 1516...
Fetching Season 1617...
Fetching Season 1718...
Fetching Season 1819...
Fetching Season 1920...
Fetching Season 2021...
Fetching Season 2122...
Fetching Season 2223...
Fetching Season 2324...
Fetching Season 2425...
Fetching Season 2526...
✅ Success! Compiled 7980 matches with full stats and odds.
Saved to: data/LaLiga_Detailed.csv
